In [ ]:
import glob
import json

import matplotlib.pyplot as plt
import numpy as np
import tifffile

from natsort import natsorted
from sklearn.linear_model import LinearRegression

%matplotlib inline
plt.style.use('seaborn-v0_8-notebook')

# A session recorded with scripts/basler_live.py, e.g. by running it with --open data/live/practical.
# To try the notebook without the camera: .venv/bin/python scripts/simulate_session.py data/live/practical
data_dir = 'data/live/practical/'

def load_tiff_directory(directory_path: str):
    tiff_paths = natsorted(glob.glob(directory_path + '*.tif*'))
    curr_img = tifffile.imread(tiff_paths[0])
    images = np.empty((len(tiff_paths), curr_img.shape[0], curr_img.shape[1]), dtype=np.float32)
    images[0] = curr_img

    for i, tiff_path in enumerate(tiff_paths[1:]):
        images[i+1] = tifffile.imread(tiff_path)
        
    return images

def load_metadata(directory_path: str):
    """The settings read back from the camera when the recording was made."""
    with open(directory_path + 'metadata.json') as f:
        return json.load(f)

**A note on units:** the recordings are the camera's raw 12-bit values (Mono12, 0-4095), stored in 16-bit TIFFs without any shift. All counts below are in these 12-bit units.

Every recording folder also holds `metadata.json`, with the gain, exposure time and black level read back from the camera, so nothing has to be inferred from folder names.

## Measure readout noise

### Data: 
A sequence of images (∼ 100) from the camera with NO light hitting the detector at an extremely short exposure time (∼ 1 ms). Recorded with the Read noise tab of `basler_live.py`, at the camera's minimum exposure.

Set the black level (bias) high enough that no pixel reads 0 before recording. Otherwise the negative fluctuations are clipped at 0 and the standard deviation comes out too low.

### Exercise instructions:
Measure the standard deviation of each pixels value across the movie (use ImageJ). What is the average noise (standard deviation in counts) across pixels?

In [ ]:
def measure_readout_noise(im_data, downsample=5):
    im_data = im_data[:, ::downsample, ::downsample]
    
    return np.std(im_data, axis=0, ddof=1, dtype=np.float32)

def calc_plot_readout_noise(im_data, axis, downsample=5):
    pix_dev = measure_readout_noise(im_data, downsample)
    mean_dev = np.mean(pix_dev, dtype=np.float32)
    
    axis.hist(pix_dev.flatten(), bins=100)
    axis.axvline(mean_dev, color='r', linestyle='--')
    axis.text(0.8, 0.8, f'Mean: {mean_dev:.3f}', ha='center', va='center', transform=axis.transAxes)
    axis.set_ylabel('Count')
    axis.set_xlabel('Standard deviation')
    
    return mean_dev

def plot_readout_noise_multiple_gain(gain_levels: list[str], path_stub: str,  downsample: int):
    num_plots = (len(gain_levels) + 1)// 2
    
    fig, ax = plt.subplots(num_plots, 2, figsize=(14, 14))
    mean_devs = []
    bias = []
    
    for i, gain in enumerate(gain_levels):
        read_noise_images = load_tiff_directory(f'{path_stub}{gain}dB/')
        bias.append(np.mean(read_noise_images, dtype=np.float32))
        print(f'{gain} dB: {np.mean(read_noise_images == 0):.1%} of values are clipped at 0')
        mean_devs.append(calc_plot_readout_noise(read_noise_images, ax[i//2][i%2], downsample))
        ax[i//2][i%2].set_title(f'{gain}dB')
        
    plt.show()
    
    fig, ax = plt.subplots()
    ax.plot(gain_levels, mean_devs, marker='o')
    ax.set_xlabel('Gain (dB)')
    ax.set_ylabel('Read noise (counts)')
    plt.show()
    
    return mean_devs, bias

Let's plot the readout noise for the Basler camera at different gain levels.

In [ ]:
gain_levels = [0, 1, 12, 24]
read_noise_stub = f'{data_dir}read_noise/'
downsample_by = 5

# Check which camera recorded the session: the simulator's recordings say "Simulated"
print(f"Camera: {load_metadata(f'{read_noise_stub}{gain_levels[0]}dB/')['camera_model']}")
read_noise, read_bias = plot_readout_noise_multiple_gain(gain_levels, read_noise_stub, downsample_by)

### Conclusions

*Fill in the numbers after the live session.*

**Check the printout first: 0% of values should be clipped at 0.** If any are, the black level was too low, so the negative fluctuations were cut off and the standard deviation is too low. Record again with a higher black level.

| Gain | Bias (counts) | Read noise (counts) |
|---|---|---|
| 0 dB | | |
| 1 dB | | |
| 12 dB | | |
| 24 dB | | |

Read noise in counts increases with gain because the amplifier also amplifies noise generated before it.

## Measure dark counts

### Data:
A sequence of images (∼ 100) from the camera with NO light hitting the detector at a range of exposure times.

### Exercise instructions:
Measure the Mean number of counts in each image. 

Do the counts increase with increasing exposure time? How many counts/pixel/second do you observe? Hint: Remember to subtract the Bias Level, the average value (in counts) appearing in the image following a 0-second exposure. (This is an offset added to each image by the camera manufacturers and does not reflect the incident light level.)

The Dark counts tab of `basler_live.py` records a short stack at each exposure, for each gain, into `dark/<gain>dB/<exposure>us/`.

In [ ]:
def measure_dark_counts(im_data):
    return np.mean(im_data, axis=(1,2), dtype=np.float32)

def calc_plot_dark_counts(im_data, exposures):
    dark_counts = measure_dark_counts(im_data)
    bias = dark_counts[0]
    subtracted_counts = dark_counts - bias
    
    model_dark_noise = LinearRegression()
    model_dark_noise.fit(exposures.reshape(-1, 1), subtracted_counts)
    
    print(f'The slope of the dark noise is {model_dark_noise.coef_}')
    print(f'The y-intercept of the dark noise is {model_dark_noise.intercept_}')
    
    fig, ax = plt.subplots()
    ax.plot(exposures, subtracted_counts, marker='o')
    ax.set_xlabel('Exposure time (s)')
    ax.set_ylabel('Mean counts')
    plt.show()
    
    return dark_counts, subtracted_counts, model_dark_noise

def load_dark_series(gain_dir):
    """The mean image at each exposure, and the exposures in seconds from metadata.json."""
    folders = sorted(glob.glob(gain_dir + '*us/'), key=lambda f: load_metadata(f)['exposure_us'])
    exposures = np.array([load_metadata(f)['exposure_us'] * 1e-6 for f in folders])
    images = np.stack([load_tiff_directory(f).mean(axis=0) for f in folders])
    return images, exposures

In [ ]:
dark_count_results = []
dark_exposures = []

for gain in gain_levels:
    print(f'Dark counts for {gain}dB')
    dark_images, exposures = load_dark_series(f'{data_dir}dark/{gain}dB/')
    dark_exposures.append(exposures)
    dark_count_results.append(calc_plot_dark_counts(dark_images, exposures))

### Conclusions

*Fill in the numbers after the live session.*

| Gain | Bias (counts) | Dark counts (counts/pixel/s) | Dark counts (e-/pixel/s) |
|---|---|---|---|
| 0 dB | | | |
| 1 dB | | | |
| 12 dB | | | |
| 24 dB | | | |

The bias is the mean at the shortest exposure, where dark counts are negligible. The fit intercepts should be close to 0 after subtracting it. Dark counts rise with gain because electrons are amplified whatever their source. Convert them to e-/pixel/s with the gain measured in the shot noise section.

## Compute the dynamic range of the camera

First determine the cameras full well capacity (saturation value) in counts after bias removal.

Use a long exposure and expose the camera to a bright light source. What value are saturated
pixels assigned? Use the formula below to compute the cameras dynamic range (the number of
individually discernible light levels) for a fixed exposure.
$$
\text{Dynamic Range} = (\text{Saturation} - \text{Dark counts}) / \text{Read-out noise}
$$
Note: Use bias subtracted values for both Saturation and Dark counts.
Compare your results to the cameras technical specification sheet.

In [ ]:
# metadata.json records the largest value the camera can output: 4095 for 12-bit Mono12.
saturated_value = load_metadata(f'{read_noise_stub}{gain_levels[0]}dB/')['max_value']
print(f'Saturated pixels are assigned the value {saturated_value}')

# A recording from the Dynamic range tab's "Record saturated frames" confirms it
for path in natsorted(glob.glob(f'{data_dir}saturation/*/')):
    print(f'{path}: brightest value {int(load_tiff_directory(path).max())}')

exposure_index = 1  # 1 ms in the default exposure list
dynamic_range = []

for i, gain in enumerate(gain_levels):
    # dark_count_results[i][0][0] is the bias for the given gain
    bias = dark_count_results[i][0][0]
    sat_value = saturated_value - bias
    dark_count = dark_count_results[i][1][exposure_index]
    dynamic_range.append((sat_value - dark_count) / read_noise[i])

    print(f'{gain} dB: saturation after bias removal is {sat_value:.0f} counts, '
          f'dark counts at {dark_exposures[i][exposure_index]*1000:.2f} ms are {dark_count:.2f}, '
          f'dynamic range is {dynamic_range[i]:.0f}:1 ({20 * np.log10(dynamic_range[i]):.1f} dB)')

### Conclusions

*Fill in the numbers after the live session.*

| Gain | Saturation after bias removal (counts) | Dynamic range | Dynamic range (dB) |
|---|---|---|---|
| 0 dB | | | |
| 1 dB | | | |
| 12 dB | | | |
| 24 dB | | | |

Expect the dynamic range to be highest at the lowest gain: increasing the gain raises read noise in counts, while the digitizer's maximum value stays fixed. The comparison with the specification sheet is at the end of the notebook.

## Measure shot noise

### Data:
Acquire sequences of images (100) with the camera exposed to a wide distribution of brightness values. You can do this in two ways:
- Use a range of pixel intensities within an image: structured sample illuminated at a fixed intensity. Measure the mean and standard deviation for each pixel across the image sequence.
- Use a range of illumination intensities: same pixel, across different LED currents. Measure the mean and standard deviation of a select pixel across the images acquired at various illumination intensities.

Use 1 second exposure time for either option. The illumination light must be stable; use an LED in an enclosed container (to avoid background changes).

Record with the Shot noise tab of `basler_live.py`. It saves the lit stack, then asks you to cover the camera and saves a dark stack at identical settings (`shot/<label>_dark`). The per-pixel mean of the dark stack is the offset, which removes the bias, the fixed pattern and the dark counts together.

### Exercise instructions:
Plot the variance, i.e. standard deviation squared (Y axis) vs. mean (X axis) for different brightness levels. Fit a line. Determine the detectors gain (counts per photoelectron):

Hint: Remember to subtract out the Bias Level measured above. 

Note: The slope of the line will be the Gain of your camera.

Where does the fit line intersect the y-axis? Does this make sense?

In [ ]:
def check_frames(directory_path):
    """The camera numbers its frames, and basler_live.py records any it missed or repeated."""
    meta = load_metadata(directory_path)
    assert meta['missing_frames'] == 0 and meta['repeated_frames'] == 0, f'{directory_path}: frames were dropped or repeated'
    return meta

def measure_shot_noise(image_path, dark_path):
    check_frames(image_path)
    offset = np.mean(load_tiff_directory(dark_path), axis=0)
    shot_image = load_tiff_directory(image_path) - offset

    # Rescale each frame to the same mean so that frame-to-frame changes in the
    # illumination are not counted as shot noise.
    frame_means = np.mean(shot_image, axis=(1, 2), dtype=np.float64)
    print(f'{shot_image.shape[0]} frames, frame means vary by {np.std(frame_means, ddof=1) / np.mean(frame_means):.2%}')
    shot_image = shot_image / (frame_means / np.mean(frame_means)).reshape(-1, 1, 1).astype(np.float32)

    mean = np.mean(shot_image, axis=0, dtype=np.float32).reshape(-1, 1)
    var = np.var(shot_image, axis=0, ddof=1, dtype=np.float32).reshape(-1, 1)

    return mean, var

def calculate_plot_shot_noise(image_path, dark_path, downsample=5):
    mean, var = measure_shot_noise(image_path, dark_path)

    model = LinearRegression()
    model.fit(mean, var)

    # The intercept is an extrapolation, so check how much it depends on the fit range
    dim_pixels = mean[:, 0] < 0.35 * mean.max()
    model_dim = LinearRegression()
    model_dim.fit(mean[dim_pixels], var[dim_pixels])

    best_fit = model.predict(mean)

    print(f'The gain of the camera is {model.coef_[0][0]:.3f} counts/e-')
    print(f'The y-intercept is {model.intercept_[0]:.3f}')
    print(f'Fitting the dimmest pixels only, the gain is {model_dim.coef_[0][0]:.3f} counts/e- and the y-intercept is {model_dim.intercept_[0]:.3f}')
    print(f'The dimmest pixel has a mean of {mean.min():.0f} counts')

    fig, ax = plt.subplots()
    ax.scatter(mean[::downsample], var[::downsample])
    ax.plot(mean, best_fit, color='r', linestyle='--')
    ax.set_xlabel('Mean counts')
    ax.set_ylabel('Variance')

    plt.show()

    return model.coef_[0][0], model.intercept_[0]

In [ ]:
# Each lit recording shot/<label>/ is paired with the dark stack shot/<label>_dark/ taken at the same settings
shot_paths = natsorted(p for p in glob.glob(f'{data_dir}shot/*/') if not p.rstrip('/').endswith('_dark'))
shot_results = []

for path in shot_paths:
    meta = load_metadata(path)
    print(f"Shot noise for {meta['label']}: {meta['gain']:g} dB, {meta['exposure_us'] / 1000:g} ms exposure")
    gain, intercept = calculate_plot_shot_noise(path, path.rstrip('/') + '_dark/')
    shot_results.append({'path': path, 'gain_db': meta['gain'], 'counts_per_e': gain, 'intercept': intercept})

In [ ]:
shot_gain_db = shot_results[0]['gain_db']
# The camera reads its gain back to its own step size (e.g. 11.99 dB for 12), so match the nearest setting
gain_index = int(np.argmin(np.abs(np.array(gain_levels) - shot_gain_db)))
assert abs(gain_levels[gain_index] - shot_gain_db) < 0.1, 'record read noise and dark counts at the shot noise gain too'
same_gain = [r for r in shot_results if abs(r['gain_db'] - shot_gain_db) < 0.1]
camera_gain = np.mean([r['counts_per_e'] for r in same_gain])
print(f"Gain at {shot_gain_db:g} dB is {camera_gain:.3f} counts/e-, or {1 / camera_gain:.2f} e- per count")

readout_noise_e = read_noise[gain_index] / camera_gain
print(f"Read noise in e- for {shot_gain_db:g} dB is: {readout_noise_e:.3f}")

for r in same_gain:
    implied = np.sqrt(r['intercept']) if r['intercept'] > 0 else np.nan
    print(f"Read noise implied by the y-intercept of {r['path']} is {implied:.2f} counts ({implied / camera_gain:.2f} e-), "
          f"compared to {read_noise[gain_index]:.2f} counts measured directly")

dark_rate = dark_count_results[gain_index][2].coef_[0]
print(f"Dark counts at {shot_gain_db:g} dB are {dark_rate:.2f} counts/pixel/s, or {dark_rate / camera_gain:.2f} e-/pixel/s")

In [ ]:
# With shot noise recorded at several gains, the measured gain should follow the nominal 10^(dB/20)
for r in shot_results:
    print(f"{r['gain_db']:g} dB: {r['counts_per_e']:.3f} counts/e-, "
          f"implying {r['counts_per_e'] / 10 ** (r['gain_db'] / 20):.3f} counts/e- at 0 dB")

# Nominal gain at 0 dB, assuming the gain setting in dB is exact
gain_0db = camera_gain / 10 ** (shot_gain_db / 20)
sat_0db = saturated_value - dark_count_results[0][0][0]
print(f'\nAt 0 dB: gain {gain_0db:.3f} counts/e-, read noise {read_noise[0] / gain_0db:.2f} e-, '
      f'saturation {sat_0db / gain_0db:.0f} e-, dynamic range {dynamic_range[0]:.0f}:1 ({20 * np.log10(dynamic_range[0]):.1f} dB)')

## Conclusions

*Fill in the numbers after the live session.*

### Shot noise

The slope of variance against mean is the gain in counts/e-. Check the printout for two things:
- **Frame means:** if they vary by more than a fraction of a percent, the light was not stable. The frames are rescaled to a common mean so that the extra variance does not inflate the slope.
- **Dropped frames:** `check_frames` stops if the camera reported missing or repeated frames, because those are not independent measurements.

| Gain setting | Gain (counts/e-) | e- per count |
|---|---|---|
| | | |

### Where does the fit line intersect the y-axis? Does this make sense?

Theory predicts `var(c) = αE[c] + var(read noise)`, so the intercept should be the read noise variance measured directly in the first section. Because the dark stack removes the bias exactly, the intercept is not shifted by an offset guess. It is still an extrapolation: the further the dimmest pixel is from 0, and the more the relationship curves, the less precise it is. Compare the all-pixel and dimmest-pixel fits.

| Fit | Intercept (counts²) | Implied read noise (e-) | Read noise measured directly (e-) |
|---|---|---|---|
| All pixels | | | |
| Dimmest pixels only | | | |

### Comparison with the specification sheet

The specification values describe the camera at its base gain, so compare them to the 0 dB results. The 0 dB gain is taken from the measured gain, assuming the gain setting in dB is exact.

| Quantity at 0 dB | Measured | Specification |
|---|---|---|
| Read noise | | 6.9 e- |
| Saturation | | 32.8 ke- |
| Dynamic range | | 73.5 dB |

The specification values are EMVA 1288 figures as listed by [a reseller](https://ww2.avsupply.com/ITM/31548/aca1920-40um.html), not taken from a Basler document.

Saturation in electrons only reflects the true full well capacity at low gain. At higher gain the digitizer reaches its maximum first.